# Análisis integral del dataset Chest X-Ray Images (Pneumonia)

Este pipeline reconstruye el proyecto completo a partir del dataset local.
La decisión de mantenerlo como script con marcas `# %%` responde a dos
objetivos: por un lado generar un notebook entregable; por otro, conservar
una base de código que sea fácil de leer, depurar y versionar.



In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import pickle
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib

# Usamos un backend sin interfaz gráfica para que el pipeline sea ejecutable
# tanto desde terminal como desde notebook sin depender de una ventana local.
matplotlib.use('Agg')


def resolve_project_root() -> Path:
    """Detecta la raíz del proyecto desde script o notebook.

    Se usa una heurística simple porque el mismo código debe correr desde
    `scripts/` y también desde `notebooks/` cuando el ejecutor lanza el `.ipynb`.
    """
    if '__file__' in globals():
        return Path(__file__).resolve().parents[1]

    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists():
        return cwd
    return cwd.parent


PROJECT_ROOT = resolve_project_root()
DEPENDENCY_CANDIDATES = [
    Path(r'C:\tmp\new_project_runtime'),
    PROJECT_ROOT / '.deps_runtime',
    PROJECT_ROOT / '.deps',
]
ACTIVE_DEPENDENCIES = []
if sys.prefix == getattr(sys, 'base_prefix', sys.prefix):
    for candidate in DEPENDENCY_CANDIDATES:
        if candidate.exists():
            ACTIVE_DEPENDENCIES.append(candidate)
            if str(candidate).lower().startswith(r'c:\tmp\new_project_runtime'.lower()):
                break

    for candidate in ACTIVE_DEPENDENCIES:
        # La ruta se agrega al final para preferir librerías sanas del entorno y,
        # si ya existe un runtime limpio, no mezclarlo con carpetas rotas.
        sys.path.append(str(candidate))

warnings.filterwarnings('ignore')
os.environ.setdefault('PYTHONHASHSEED', '42')
os.environ.setdefault('TORCH_HOME', str(PROJECT_ROOT / 'artifacts' / 'models' / 'torch_cache'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, models, transforms
from torchvision.models import DenseNet121_Weights

try:
    from IPython.display import display
except Exception:  # pragma: no cover - fallback para ejecución por terminal
    def display(obj):
        print(obj)


ImageFile.LOAD_TRUNCATED_IMAGES = True
sns.set_theme(style='whitegrid', context='talk')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Elegimos un dataclass porque concentra la configuración operativa en un solo
# lugar y evita "números mágicos" dispersos por el pipeline.
@dataclass
class RunConfig:
    profile: str
    batch_size: int
    image_size: int
    anomaly_size: int
    train_cap: int | None
    val_cap: int | None
    test_cap: int | None
    pca_sample_size: int
    histogram_sample_size: int
    autoencoder_normal_train_limit: int
    autoencoder_eval_limit: int
    autoencoder_epochs: int
    cnn_epochs: int
    transfer_head_epochs: int
    transfer_finetune_epochs: int
    learning_rate: float
    fine_tune_learning_rate: float
    early_stopping_patience: int
    isolation_contamination: float


PROFILE = os.getenv('RUN_PROFILE', 'full').strip().lower()
if PROFILE not in {'verify', 'full'}:
    raise ValueError("RUN_PROFILE must be 'verify' or 'full'")

CONFIG = RunConfig(
    profile=PROFILE,
    batch_size=16,
    image_size=224,
    anomaly_size=64,
    train_cap=1024 if PROFILE == 'verify' else None,
    val_cap=256 if PROFILE == 'verify' else None,
    test_cap=256 if PROFILE == 'verify' else None,
    pca_sample_size=1500 if PROFILE == 'verify' else 5000,
    histogram_sample_size=300 if PROFILE == 'verify' else 800,
    autoencoder_normal_train_limit=512 if PROFILE == 'verify' else 1400,
    autoencoder_eval_limit=512 if PROFILE == 'verify' else 1500,
    autoencoder_epochs=2 if PROFILE == 'verify' else 6,
    cnn_epochs=2 if PROFILE == 'verify' else 8,
    transfer_head_epochs=1 if PROFILE == 'verify' else 3,
    transfer_finetune_epochs=1 if PROFILE == 'verify' else 3,
    learning_rate=1e-3,
    fine_tune_learning_rate=1e-4,
    early_stopping_patience=2 if PROFILE == 'verify' else 3,
    isolation_contamination=0.03,
)

DATA_ROOT = PROJECT_ROOT / 'data'
RAW_ROOT = DATA_ROOT / 'raw' / 'chest_xray'
INTERIM_ROOT = DATA_ROOT / 'interim'
PROCESSED_ROOT = DATA_ROOT / 'processed' / 'split_801010'
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = ARTIFACTS_ROOT / 'figures'
METRICS_DIR = ARTIFACTS_ROOT / 'metrics'
MODELS_DIR = ARTIFACTS_ROOT / 'models'
REPORTS_DIR = ARTIFACTS_ROOT / 'reports'
TENSORBOARD_DIR = METRICS_DIR / 'tensorboard'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

for path in [
    INTERIM_ROOT,
    PROCESSED_ROOT,
    FIGURES_DIR,
    METRICS_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    TENSORBOARD_DIR,
    CONFIGS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f'No se encontró el dataset en {RAW_ROOT}. Debe existir antes de ejecutar el pipeline.'
    )

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Run profile: {CONFIG.profile}')



Project root: C:\Users\TeaDyDy\Documents\New project
Device: cpu
Run profile: verify


## Utilidades de preprocesamiento y metadatos

Esta sección existe para aislar la lógica del dataset. La decisión de separar
aquí la indexación, validación y re-partición evita mezclar preocupación de
datos con preocupación de modelado.



In [16]:
def load_grayscale_array(image_path: Path, size: int) -> np.ndarray:
    with Image.open(image_path) as image:
        return np.asarray(image.convert('L').resize((size, size)), dtype=np.float32) / 255.0


def load_rgb_image(image_path: Path) -> Image.Image:
    with Image.open(image_path) as image:
        return image.convert('RGB')


def build_train_transform(image_size: int) -> transforms.Compose:
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    return transforms.Compose(
        [
            transforms.Lambda(lambda img: img.convert('RGB')),
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ]
    )


def build_eval_transform(image_size: int) -> transforms.Compose:
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    return transforms.Compose(
        [
            transforms.Lambda(lambda img: img.convert('RGB')),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ]
    )


def build_metadata(raw_root: Path) -> pd.DataFrame:
    """Indexa el dataset crudo sin modificarlo.

    La razón de crear una tabla explícita es que facilita trazabilidad:
    podemos saber de qué split original salió cada imagen y conservar un
    `sample_id` estable para reportes y cruces posteriores.
    """
    records: list[dict] = []
    for original_split in ['train', 'val', 'test']:
        split_dir = raw_root / original_split
        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            label_name = class_dir.name.upper()
            label = 1 if label_name == 'PNEUMONIA' else 0
            for image_path in sorted(class_dir.glob('*')):
                if image_path.suffix.lower() not in {'.jpeg', '.jpg', '.png'}:
                    continue
                records.append(
                    {
                        'source_split': original_split,
                        'label_name': label_name,
                        'label': label,
                        'image_path': str(image_path.resolve()),
                        'image_name': image_path.name,
                    }
                )

    metadata = pd.DataFrame(records)
    if metadata.empty:
        raise RuntimeError('No se encontraron imágenes en la estructura original del dataset.')

    metadata['sample_id'] = [f'img_{idx:05d}' for idx in range(len(metadata))]
    metadata.to_csv(INTERIM_ROOT / 'original_metadata.csv', index=False)
    return metadata


def create_resplit(metadata: pd.DataFrame, output_root: Path, seed: int = 42) -> pd.DataFrame:
    """Genera un split 80/10/10 reproducible.

    Copiar archivos a `processed/` en vez de mutar el dataset crudo hace el
    pipeline más seguro: la materia prima queda intacta y la preparación pasa
    a ser un artefacto reproducible, no una operación destructiva.
    """
    manifest_path = INTERIM_ROOT / 'split_manifest.csv'
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        expected_columns = {'sample_id', 'image_path', 'label_name', 'label', 'split_path'}
        if expected_columns.issubset(set(manifest.columns)):
            return manifest

    base = metadata[['sample_id', 'image_path', 'label_name', 'label']].copy()
    train_df, temp_df = train_test_split(
        base,
        test_size=0.2,
        random_state=seed,
        stratify=base['label'],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=seed,
        stratify=temp_df['label'],
    )

    split_frames = {'train': train_df, 'val': val_df, 'test': test_df}
    manifest_rows: list[dict] = []

    for split_name, frame in split_frames.items():
        for row in frame.itertuples(index=False):
            src = Path(row.image_path)
            destination_dir = output_root / split_name / row.label_name
            destination_dir.mkdir(parents=True, exist_ok=True)
            destination = destination_dir / f'{row.sample_id}_{src.name}'
            if not destination.exists():
                shutil.copy2(src, destination)
            manifest_rows.append(
                {
                    'sample_id': row.sample_id,
                    'image_path': row.image_path,
                    'label_name': row.label_name,
                    'label': row.label,
                    'split_name': split_name,
                    'split_path': str(destination.resolve()),
                }
            )

    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(manifest_path, index=False)
    return manifest


def verify_images(metadata: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Verifica integridad básica de archivos de imagen."""
    valid_records = []
    invalid_records = []
    for row in metadata.itertuples(index=False):
        image_path = Path(row.split_path)
        try:
            with Image.open(image_path) as image:
                image.verify()
            valid_records.append(row._asdict())
        except Exception as exc:  # pragma: no cover - ruta defensiva
            invalid_records.append({**row._asdict(), 'error': str(exc)})

    valid_df = pd.DataFrame(valid_records)
    invalid_df = pd.DataFrame(invalid_records)
    if not invalid_df.empty:
        invalid_df.to_csv(REPORTS_DIR / 'invalid_images.csv', index=False)
    return valid_df, invalid_df


def sample_rows(frame: pd.DataFrame, max_rows: int | None, seed: int = 42) -> pd.DataFrame:
    """Recorta filas manteniendo mezcla razonable por clase.

    En modo `verify` priorizamos tiempo de feedback sobre exhaustividad.
    El criterio es tomar una muestra estratificada aproximada para no perder
    completamente la forma del problema.
    """
    if max_rows is None or len(frame) <= max_rows:
        return frame.copy()

    groups = []
    for _, group in frame.groupby('label_name', group_keys=False):
        proportion = len(group) / len(frame)
        target = max(1, int(round(max_rows * proportion)))
        groups.append(group.sample(n=min(target, len(group)), random_state=seed))

    sampled = pd.concat(groups).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return sampled.iloc[:max_rows].copy()


def compute_intensity_statistics(metadata: pd.DataFrame, size: int) -> pd.DataFrame:
    rows = []
    for row in metadata.itertuples(index=False):
        array = load_grayscale_array(Path(row.split_path), size=size)
        rows.append(
            {
                'sample_id': row.sample_id,
                'split_name': row.split_name,
                'label_name': row.label_name,
                'label': row.label,
                'mean_intensity': float(array.mean()),
                'std_intensity': float(array.std()),
                'l2_norm': float(np.linalg.norm(array)),
            }
        )
    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(REPORTS_DIR / 'image_intensity_statistics.csv', index=False)
    return stats_df




## Visualización y detección de anomalías

Las gráficas y detectores simples no sustituyen la revisión clínica, pero sí
ayudan a encontrar sesgos, muestras atípicas y errores de calidad antes de
entrenar modelos más costosos.



In [17]:
def plot_class_distribution(frame: pd.DataFrame, output_path: Path, title: str) -> None:
    plt.figure(figsize=(8, 5))
    ax = sns.countplot(data=frame, x='label_name', hue='label_name', palette='viridis', legend=False)
    ax.set_title(title)
    ax.set_xlabel('Clase')
    ax.set_ylabel('Cantidad de imágenes')
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()


def plot_sample_images(frame: pd.DataFrame, output_path: Path, n_per_class: int = 3) -> None:
    samples = pd.concat(
        [
            group.sample(n=min(n_per_class, len(group)), random_state=SEED)
            for _, group in frame.groupby('label_name', group_keys=False)
        ],
        ignore_index=True,
    )

    cols = n_per_class
    rows = samples['label_name'].nunique()
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_2d(axes)

    for row_idx, (label_name, group) in enumerate(samples.groupby('label_name')):
        for col_idx in range(cols):
            ax = axes[row_idx, col_idx]
            ax.axis('off')
            if col_idx < len(group):
                image_path = Path(group.iloc[col_idx]['split_path'])
                ax.imshow(load_rgb_image(image_path), cmap='gray')
                ax.set_title(label_name)

    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()


def plot_histograms(frame: pd.DataFrame, output_path: Path, size: int) -> None:
    sampled = sample_rows(frame, CONFIG.histogram_sample_size, seed=SEED)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, label_name in zip(axes, ['NORMAL', 'PNEUMONIA']):
        subset = sampled[sampled['label_name'] == label_name]
        if subset.empty:
            continue
        for row in subset.itertuples(index=False):
            array = load_grayscale_array(Path(row.split_path), size=size).ravel()
            ax.hist(array, bins=40, alpha=0.08, density=True, color='steelblue')
        ax.set_title(f'Histograma de intensidades: {label_name}')
        ax.set_xlabel('Intensidad normalizada')
    axes[0].set_ylabel('Densidad')
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()


def run_pca_isolation_forest(frame: pd.DataFrame, size: int) -> pd.DataFrame:
    sampled = sample_rows(frame, CONFIG.pca_sample_size, seed=SEED).reset_index(drop=True)
    matrix = np.stack([load_grayscale_array(Path(path), size=size).ravel() for path in sampled['split_path']])
    components = min(50, matrix.shape[0] - 1, matrix.shape[1])
    pca = PCA(n_components=components, svd_solver='randomized', random_state=SEED)
    embedding = pca.fit_transform(matrix)

    iso = IsolationForest(
        n_estimators=200,
        contamination=CONFIG.isolation_contamination,
        random_state=SEED,
    )
    predictions = iso.fit_predict(embedding)
    scores = iso.decision_function(embedding)

    result = sampled.copy()
    result['pca_outlier'] = predictions == -1
    result['isolation_score'] = scores
    result.sort_values('isolation_score').to_csv(
        REPORTS_DIR / 'pca_isolation_forest_results.csv',
        index=False,
    )

    plt.figure(figsize=(10, 8))
    plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=result['label'],
        cmap='coolwarm',
        alpha=0.6,
        s=25,
    )
    plt.scatter(
        embedding[result['pca_outlier'].to_numpy(), 0],
        embedding[result['pca_outlier'].to_numpy(), 1],
        facecolors='none',
        edgecolors='black',
        linewidths=1.2,
        s=80,
    )
    plt.title('PCA + Isolation Forest')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'pca_isolation_forest.png', dpi=180)
    plt.close()
    return result




## Datasets, modelos y entrenamiento

La lógica de entrenamiento se encapsula en funciones reutilizables. La
intención aquí es reducir duplicación entre la CNN base y el modelo con
transfer learning, manteniendo una sola forma de registrar métricas.



In [18]:
class MetadataTensorDataset(Dataset):
    """Dataset simple para modelos que consumen tensores ya normalizados."""

    def __init__(self, frame: pd.DataFrame, image_size: int, rgb: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.image_size = image_size
        self.rgb = rgb

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        image_path = Path(row['split_path'])
        with Image.open(image_path) as image:
            if self.rgb:
                image = image.convert('RGB').resize((self.image_size, self.image_size))
                array = np.asarray(image, dtype=np.float32) / 255.0
                tensor = torch.from_numpy(np.transpose(array, (2, 0, 1)))
            else:
                image = image.convert('L').resize((self.image_size, self.image_size))
                array = np.asarray(image, dtype=np.float32) / 255.0
                tensor = torch.from_numpy(array).unsqueeze(0)

        label = torch.tensor(float(row['label']), dtype=torch.float32)
        return tensor, label, row['sample_id']


class SimpleAutoencoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 1, kernel_size=2, stride=2),
            nn.Sigmoid(),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(inputs))


class BaseCNN(nn.Module):
    """CNN pequeña para tener una línea base interpretable y barata."""

    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 1),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(inputs))


def build_imagefolder_datasets(processed_root: Path, image_size: int):
    train_transform = build_train_transform(image_size)
    eval_transform = build_eval_transform(image_size)
    train_ds = datasets.ImageFolder(processed_root / 'train', transform=train_transform)
    val_ds = datasets.ImageFolder(processed_root / 'val', transform=eval_transform)
    test_ds = datasets.ImageFolder(processed_root / 'test', transform=eval_transform)
    return train_ds, val_ds, test_ds


def subset_dataset(dataset: datasets.ImageFolder, max_size: int | None, seed: int = 42) -> Dataset:
    if max_size is None or len(dataset) <= max_size:
        return dataset

    targets = np.array(dataset.targets)
    indices = np.arange(len(dataset))
    selected: list[int] = []

    for class_id in np.unique(targets):
        class_indices = indices[targets == class_id]
        proportion = len(class_indices) / len(indices)
        target_n = max(1, int(round(max_size * proportion)))
        rng = np.random.default_rng(seed + int(class_id))
        chosen = rng.choice(class_indices, size=min(target_n, len(class_indices)), replace=False)
        selected.extend(chosen.tolist())

    selected = sorted(selected[:max_size])
    return Subset(dataset, selected)


def compute_pos_weight(dataset: Dataset) -> torch.Tensor:
    if isinstance(dataset, Subset):
        labels = [dataset.dataset.targets[idx] for idx in dataset.indices]
    else:
        labels = dataset.targets
    positives = float(sum(labels))
    negatives = float(len(labels) - positives)
    return torch.tensor(negatives / max(positives, 1.0), dtype=torch.float32)


def build_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=CONFIG.batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


def train_autoencoder(frame: pd.DataFrame, image_size: int) -> pd.DataFrame:
    normal_train = frame[(frame['split_name'] == 'train') & (frame['label_name'] == 'NORMAL')]
    normal_train = sample_rows(normal_train, CONFIG.autoencoder_normal_train_limit, seed=SEED)
    eval_frame = sample_rows(frame, CONFIG.autoencoder_eval_limit, seed=SEED)

    train_dataset = MetadataTensorDataset(normal_train, image_size=image_size, rgb=False)
    eval_dataset = MetadataTensorDataset(eval_frame, image_size=image_size, rgb=False)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=0)

    model = SimpleAutoencoder().to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = []
    for epoch in range(CONFIG.autoencoder_epochs):
        model.train()
        running_loss = 0.0
        for inputs, _, _ in train_loader:
            inputs = inputs.to(DEVICE)
            optimizer.zero_grad()
            recon = model(inputs)
            loss = criterion(recon, inputs)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / max(len(train_dataset), 1)
        history.append({'epoch': epoch + 1, 'train_loss': epoch_loss})
        print(f'Autoencoder epoch {epoch + 1}/{CONFIG.autoencoder_epochs}: loss={epoch_loss:.4f}')

    pd.DataFrame(history).to_csv(METRICS_DIR / 'autoencoder_history.csv', index=False)

    model.eval()
    anomaly_rows = []
    with torch.no_grad():
        for inputs, labels, sample_ids in eval_loader:
            inputs = inputs.to(DEVICE)
            recon = model(inputs)
            errors = torch.mean((recon - inputs) ** 2, dim=(1, 2, 3)).cpu().numpy()
            for sample_id, label, error in zip(sample_ids, labels.numpy(), errors):
                anomaly_rows.append(
                    {
                        'sample_id': sample_id,
                        'label': int(label),
                        'reconstruction_error': float(error),
                    }
                )

    anomaly_df = pd.DataFrame(anomaly_rows).merge(
        eval_frame[['sample_id', 'label_name', 'split_path']],
        on='sample_id',
        how='left',
    )
    anomaly_df.sort_values('reconstruction_error', ascending=False).to_csv(
        REPORTS_DIR / 'autoencoder_reconstruction_errors.csv',
        index=False,
    )

    plt.figure(figsize=(8, 5))
    sns.histplot(data=anomaly_df, x='reconstruction_error', hue='label_name', bins=30, kde=True)
    plt.title('Errores de reconstrucción del autoencoder')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'autoencoder_reconstruction_errors.png', dpi=180)
    plt.close()

    torch.save(model.state_dict(), MODELS_DIR / 'autoencoder_normal.pt')
    return anomaly_df


def train_binary_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    patience: int,
    experiment_name: str,
) -> tuple[nn.Module, pd.DataFrame]:
    writer = SummaryWriter(log_dir=str(TENSORBOARD_DIR / experiment_name))
    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = math.inf
    wait = 0
    history_rows = []

    for epoch in range(epochs):
        start = time.time()
        model.train()
        train_loss = 0.0
        train_targets = []
        train_probs = []

        for inputs, targets in train_loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE).float().unsqueeze(1)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
            train_probs.extend(probs.tolist())
            train_targets.extend(targets.cpu().numpy().ravel().tolist())
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        train_preds = (np.array(train_probs) >= 0.5).astype(int)
        train_acc = accuracy_score(train_targets, train_preds)

        model.eval()
        val_loss = 0.0
        val_targets = []
        val_probs = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(DEVICE)
                targets = targets.to(DEVICE).float().unsqueeze(1)
                logits = model(inputs)
                loss = criterion(logits, targets)
                probs = torch.sigmoid(logits).cpu().numpy().ravel()
                val_probs.extend(probs.tolist())
                val_targets.extend(targets.cpu().numpy().ravel().tolist())
                val_loss += loss.item() * inputs.size(0)

        val_loss /= len(val_loader.dataset)
        val_preds = (np.array(val_probs) >= 0.5).astype(int)
        val_acc = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds, zero_division=0)
        duration = time.time() - start

        writer.add_scalar('loss/train', train_loss, epoch + 1)
        writer.add_scalar('loss/val', val_loss, epoch + 1)
        writer.add_scalar('accuracy/train', train_acc, epoch + 1)
        writer.add_scalar('accuracy/val', val_acc, epoch + 1)

        history_rows.append(
            {
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_accuracy': train_acc,
                'val_accuracy': val_acc,
                'val_f1': val_f1,
                'duration_seconds': duration,
            }
        )

        print(
            f'{experiment_name} epoch {epoch + 1}/{epochs} | '
            f'train_loss={train_loss:.4f} val_loss={val_loss:.4f} '
            f'val_acc={val_acc:.4f} val_f1={val_f1:.4f}'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping activado en {experiment_name}.')
                break

    writer.close()
    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(METRICS_DIR / f'{experiment_name}_history.csv', index=False)
    return model, history_df


def collect_predictions(model: nn.Module, loader: DataLoader) -> dict[str, np.ndarray]:
    model.eval()
    logits_list = []
    probs_list = []
    targets_list = []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(DEVICE)
            logits = model(inputs)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            logits_list.append(logits.cpu().numpy().ravel())
            probs_list.append(probs)
            targets_list.append(targets.numpy().ravel())
    return {
        'logits': np.concatenate(logits_list),
        'probs': np.concatenate(probs_list),
        'targets': np.concatenate(targets_list).astype(int),
    }


def select_threshold(y_true: np.ndarray, y_score: np.ndarray) -> float:
    thresholds = np.linspace(0.1, 0.9, 81)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        preds = (y_score >= threshold).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold


def summarise_metrics(name: str, y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> dict:
    preds = (y_score >= threshold).astype(int)
    summary = {
        'model_name': name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, preds),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1_score': f1_score(y_true, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
    }
    print(f'\n{name} classification report')
    print(classification_report(y_true, preds, digits=4))
    return summary


def plot_training_history(history_df: pd.DataFrame, name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
    axes[0].set_title(f'{name} - Loss')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train_accuracy')
    axes[1].plot(history_df['epoch'], history_df['val_accuracy'], label='val_accuracy')
    axes[1].set_title(f'{name} - Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{name}_training_history.png', dpi=180)
    plt.close()


def plot_confusion_and_roc(name: str, y_true: np.ndarray, y_score: np.ndarray, threshold: float) -> None:
    preds = (y_score >= threshold).astype(int)
    cm = confusion_matrix(y_true, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NORMAL', 'PNEUMONIA'])
    disp.plot(cmap='Blues', colorbar=False)
    plt.title(f'Matriz de confusión - {name}')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{name}_confusion_matrix.png', dpi=180)
    plt.close()

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_value = roc_auc_score(y_true, y_score)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f'AUC={auc_value:.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Curva ROC - {name}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{name}_roc_curve.png', dpi=180)
    plt.close()


def export_model_bundle(
    model_name: str,
    model: nn.Module,
    threshold: float,
    metrics: dict,
    class_to_idx: dict,
    calibrator: LogisticRegression | None = None,
) -> None:
    torch.save(
        {
            'state_dict': model.state_dict(),
            'threshold': threshold,
            'metrics': metrics,
            'class_to_idx': class_to_idx,
            'run_config': asdict(CONFIG),
        },
        MODELS_DIR / f'{model_name}.pt',
    )
    if calibrator is not None:
        with open(MODELS_DIR / f'{model_name}_calibrator.pkl', 'wb') as file:
            pickle.dump(calibrator, file)


def save_experiment_log(entries: list[dict]) -> None:
    payload = {
        'project': 'chest_xray_pneumonia',
        'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
        'run_profile': CONFIG.profile,
        'device': str(DEVICE),
        'config': asdict(CONFIG),
        'experiments': entries,
    }
    with open(METRICS_DIR / 'experiment_log.json', 'w', encoding='utf-8') as file:
        json.dump(payload, file, indent=2)


def dataset_records(dataset: Dataset) -> pd.DataFrame:
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        indices = list(dataset.indices)
    else:
        base_dataset = dataset
        indices = list(range(len(base_dataset.samples)))

    rows = []
    for position, idx in enumerate(indices):
        path, label = base_dataset.samples[idx]
        rows.append(
            {
                'position': position,
                'path': path,
                'file_name': Path(path).name,
                'label': int(label),
                'label_name': base_dataset.classes[label],
            }
        )
    return pd.DataFrame(rows)


def build_prediction_frame(dataset: Dataset, predictions: dict[str, np.ndarray], threshold: float) -> pd.DataFrame:
    frame = dataset_records(dataset)
    frame['logit'] = predictions['logits']
    frame['probability'] = predictions['probs']
    frame['predicted_label'] = (frame['probability'] >= threshold).astype(int)
    frame['predicted_label_name'] = frame['predicted_label'].map({0: 'NORMAL', 1: 'PNEUMONIA'})
    frame['correct'] = frame['predicted_label'] == frame['label']
    return frame




## Explicabilidad visual

Se usa Grad-CAM porque responde exactamente a la pregunta del usuario: qué
zonas de la radiografía influyen en la decisión del modelo. No es prueba
clínica, pero sí una evidencia visual útil para inspección técnica.



In [19]:
def get_gradcam_target_layer(model_name: str, model: nn.Module) -> nn.Module:
    if model_name == 'base_cnn':
        return model.features[6]
    if model_name == 'densenet121_transfer':
        return model.features.denseblock4.denselayer16.conv2
    raise ValueError(f'No target layer configured for {model_name}')


def disable_inplace_relu(module: nn.Module) -> None:
    """Evita errores de autograd al usar hooks en capas con ReLU in-place."""
    for child_name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, child_name, nn.ReLU(inplace=False))
        else:
            disable_inplace_relu(child)


def compute_gradcam(model: nn.Module, input_tensor: torch.Tensor, target_layer: nn.Module) -> np.ndarray:
    activations = []
    gradients = []

    def forward_hook(_, __, output):
        activations.append(output)
        output.register_hook(lambda grad: gradients.append(grad))

    forward_handle = target_layer.register_forward_hook(forward_hook)
    try:
        model.eval()
        model.zero_grad(set_to_none=True)
        logits = model(input_tensor)
        logits[:, 0].backward(torch.ones_like(logits[:, 0]))

        activation = activations[-1].detach()
        gradient = gradients[-1].detach()
        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * activation).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    finally:
        forward_handle.remove()


def render_explainability_comparison(
    model_name: str,
    model: nn.Module,
    prediction_frame: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    disable_inplace_relu(model)
    target_layer = get_gradcam_target_layer(model_name, model)
    eval_transform = build_eval_transform(CONFIG.image_size)

    normal_candidates = prediction_frame[
        (prediction_frame['label'] == 0) & (prediction_frame['correct'])
    ].sort_values('probability', ascending=True)
    pneumonia_candidates = prediction_frame[
        (prediction_frame['label'] == 1) & (prediction_frame['correct'])
    ].sort_values('probability', ascending=False)

    selected_rows = []
    if not normal_candidates.empty:
        selected_rows.append(normal_candidates.iloc[0])
    if not pneumonia_candidates.empty:
        selected_rows.append(pneumonia_candidates.iloc[0])
    if len(selected_rows) < 2:
        fallback = prediction_frame.sort_values('probability', ascending=False).head(2)
        selected_rows = [row for _, row in fallback.iterrows()]

    selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)
    selected_df.to_csv(REPORTS_DIR / f'{model_name}_explainability_samples.csv', index=False)

    fig, axes = plt.subplots(len(selected_df), 3, figsize=(15, 5 * len(selected_df)))
    axes = np.atleast_2d(axes)

    for row_idx, row in selected_df.iterrows():
        image_path = Path(row['path'])
        original = load_rgb_image(image_path)
        tensor = eval_transform(original).unsqueeze(0).to(DEVICE)
        cam = compute_gradcam(model, tensor, target_layer)

        cam_image = Image.fromarray(np.uint8(cam * 255)).resize(original.size, Image.BILINEAR)
        cam_array = np.asarray(cam_image, dtype=np.float32) / 255.0
        heatmap = plt.get_cmap('jet')(cam_array)[..., :3]
        original_array = np.asarray(original, dtype=np.float32) / 255.0
        overlay = np.clip(0.58 * original_array + 0.42 * heatmap, 0.0, 1.0)

        axes[row_idx, 0].imshow(original)
        axes[row_idx, 0].set_title(f"Original\n{row['label_name']}")
        axes[row_idx, 1].imshow(cam_array, cmap='jet')
        axes[row_idx, 1].set_title('Mapa de activación')
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title(
            f"Overlay\nPred={row['predicted_label_name']} | p={row['probability']:.3f}"
        )

        for axis in axes[row_idx]:
            axis.axis('off')

    plt.suptitle(f'Grad-CAM comparativo del modelo {model_name}', y=0.99, fontsize=18)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close()
    return selected_df




## Ejecución principal del pipeline

Mantengo esta parte como una función única para que sea fácil de invocar
desde terminal, notebook o futuras automatizaciones sin duplicar lógica.



In [20]:
def run_pipeline() -> dict:
    metadata = build_metadata(RAW_ROOT)
    manifest = create_resplit(metadata, PROCESSED_ROOT, seed=SEED)
    clean_manifest, invalid_images = verify_images(manifest)

    class_counts = clean_manifest.groupby(['split_name', 'label_name']).size().reset_index(name='count')
    class_counts.to_csv(REPORTS_DIR / 'split_class_distribution.csv', index=False)
    display(class_counts)

    plot_class_distribution(clean_manifest, FIGURES_DIR / 'class_distribution.png', 'Distribución por clase y split')
    plot_sample_images(clean_manifest[clean_manifest['split_name'] == 'train'], FIGURES_DIR / 'sample_images.png')
    plot_histograms(clean_manifest, FIGURES_DIR / 'intensity_histograms.png', size=CONFIG.anomaly_size)

    image_stats = compute_intensity_statistics(clean_manifest, size=CONFIG.anomaly_size)
    extreme_intensity_rows = pd.concat(
        [
            image_stats.nsmallest(10, 'mean_intensity'),
            image_stats.nlargest(10, 'mean_intensity'),
            image_stats.nlargest(10, 'l2_norm'),
        ]
    ).drop_duplicates()
    extreme_intensity_rows.to_csv(REPORTS_DIR / 'intensity_extremes.csv', index=False)

    print(f'Imágenes válidas: {len(clean_manifest)}')
    print(f'Imágenes inválidas: {len(invalid_images)}')

    pca_outliers = run_pca_isolation_forest(clean_manifest, size=CONFIG.anomaly_size)
    display(pca_outliers.sort_values('isolation_score').head(15)[['sample_id', 'label_name', 'split_name', 'isolation_score']])

    autoencoder_outliers = train_autoencoder(clean_manifest, image_size=CONFIG.anomaly_size)
    display(
        autoencoder_outliers.sort_values('reconstruction_error', ascending=False).head(15)[
            ['sample_id', 'label_name', 'reconstruction_error']
        ]
    )

    train_dataset, val_dataset, test_dataset = build_imagefolder_datasets(PROCESSED_ROOT, CONFIG.image_size)
    train_dataset = subset_dataset(train_dataset, CONFIG.train_cap, seed=SEED)
    val_dataset = subset_dataset(val_dataset, CONFIG.val_cap, seed=SEED)
    test_dataset = subset_dataset(test_dataset, CONFIG.test_cap, seed=SEED)

    train_loader = build_loader(train_dataset, shuffle=True)
    val_loader = build_loader(val_dataset, shuffle=False)
    test_loader = build_loader(test_dataset, shuffle=False)

    pos_weight = compute_pos_weight(train_dataset).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    base_model = BaseCNN().to(DEVICE)
    base_optimizer = torch.optim.Adam(base_model.parameters(), lr=CONFIG.learning_rate)
    base_model, base_history = train_binary_classifier(
        model=base_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=base_optimizer,
        epochs=CONFIG.cnn_epochs,
        patience=CONFIG.early_stopping_patience,
        experiment_name='base_cnn',
    )
    plot_training_history(base_history, 'base_cnn')

    base_val_predictions = collect_predictions(base_model, val_loader)
    base_test_predictions = collect_predictions(base_model, test_loader)
    base_threshold = select_threshold(base_val_predictions['targets'], base_val_predictions['probs'])
    base_metrics = summarise_metrics(
        'base_cnn',
        base_test_predictions['targets'],
        base_test_predictions['probs'],
        base_threshold,
    )
    plot_confusion_and_roc('base_cnn', base_test_predictions['targets'], base_test_predictions['probs'], base_threshold)
    export_model_bundle(
        'base_cnn',
        base_model,
        threshold=base_threshold,
        metrics=base_metrics,
        class_to_idx={'NORMAL': 0, 'PNEUMONIA': 1},
    )

    transfer_weights = DenseNet121_Weights.IMAGENET1K_V1
    transfer_model = models.densenet121(weights=transfer_weights)
    for parameter in transfer_model.features.parameters():
        parameter.requires_grad = False
    transfer_model.classifier = nn.Linear(transfer_model.classifier.in_features, 1)
    transfer_model = transfer_model.to(DEVICE)

    head_optimizer = torch.optim.Adam(transfer_model.classifier.parameters(), lr=CONFIG.learning_rate)
    transfer_model, transfer_head_history = train_binary_classifier(
        model=transfer_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=head_optimizer,
        epochs=CONFIG.transfer_head_epochs,
        patience=CONFIG.early_stopping_patience,
        experiment_name='densenet121_head',
    )

    for name, parameter in transfer_model.features.named_parameters():
        if name.startswith('denseblock4') or name.startswith('norm5'):
            parameter.requires_grad = True

    fine_tune_params = [param for param in transfer_model.parameters() if param.requires_grad]
    fine_tune_optimizer = torch.optim.Adam(fine_tune_params, lr=CONFIG.fine_tune_learning_rate)
    transfer_model, transfer_finetune_history = train_binary_classifier(
        model=transfer_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=fine_tune_optimizer,
        epochs=CONFIG.transfer_finetune_epochs,
        patience=CONFIG.early_stopping_patience,
        experiment_name='densenet121_finetune',
    )

    transfer_history = pd.concat([transfer_head_history, transfer_finetune_history], ignore_index=True)
    if not transfer_history.empty:
        transfer_history['epoch'] = np.arange(1, len(transfer_history) + 1)
        plot_training_history(transfer_history, 'densenet121_transfer')

    transfer_val_predictions = collect_predictions(transfer_model, val_loader)
    transfer_test_predictions = collect_predictions(transfer_model, test_loader)
    transfer_threshold = select_threshold(transfer_val_predictions['targets'], transfer_val_predictions['probs'])

    calibrator = LogisticRegression(random_state=SEED)
    calibrator.fit(transfer_val_predictions['logits'].reshape(-1, 1), transfer_val_predictions['targets'])
    calibrated_test_probs = calibrator.predict_proba(transfer_test_predictions['logits'].reshape(-1, 1))[:, 1]

    transfer_metrics = summarise_metrics(
        'densenet121_transfer',
        transfer_test_predictions['targets'],
        calibrated_test_probs,
        transfer_threshold,
    )
    plot_confusion_and_roc(
        'densenet121_transfer',
        transfer_test_predictions['targets'],
        calibrated_test_probs,
        transfer_threshold,
    )
    export_model_bundle(
        'densenet121_transfer',
        transfer_model,
        threshold=transfer_threshold,
        metrics=transfer_metrics,
        class_to_idx=train_dataset.dataset.class_to_idx if isinstance(train_dataset, Subset) else train_dataset.class_to_idx,
        calibrator=calibrator,
    )

    metrics_summary = pd.DataFrame([base_metrics, transfer_metrics]).sort_values('f1_score', ascending=False)
    metrics_summary.to_csv(METRICS_DIR / 'metrics_summary.csv', index=False)
    display(metrics_summary)

    base_prediction_frame = build_prediction_frame(test_dataset, base_test_predictions, threshold=base_threshold)
    base_prediction_frame.to_csv(REPORTS_DIR / 'base_cnn_test_predictions.csv', index=False)

    transfer_prediction_frame = build_prediction_frame(
        test_dataset,
        {
            'logits': transfer_test_predictions['logits'],
            'probs': calibrated_test_probs,
            'targets': transfer_test_predictions['targets'],
        },
        threshold=transfer_threshold,
    )
    transfer_prediction_frame.to_csv(REPORTS_DIR / 'densenet121_transfer_test_predictions.csv', index=False)

    best_model_name = metrics_summary.iloc[0]['model_name']
    best_model_path = MODELS_DIR / f'{best_model_name}.pt'
    best_model = base_model if best_model_name == 'base_cnn' else transfer_model
    best_prediction_frame = base_prediction_frame if best_model_name == 'base_cnn' else transfer_prediction_frame
    explainability_path = FIGURES_DIR / f'{best_model_name}_gradcam_comparison.png'
    render_explainability_comparison(
        model_name=best_model_name,
        model=best_model,
        prediction_frame=best_prediction_frame,
        output_path=explainability_path,
    )

    api_contract = {
        'route': '/predict',
        'method': 'POST',
        'request_content_type': 'multipart/form-data',
        'request_fields': {'file': 'JPEG/PNG chest X-ray image'},
        'response_schema': {
            'predicted_label': 'NORMAL|PNEUMONIA',
            'pneumonia_probability': 'float',
            'threshold': 'float',
            'model_name': 'string',
        },
        'preprocessing': {
            'image_mode': 'RGB',
            'image_size': CONFIG.image_size,
            'normalization': 'ImageNet mean/std',
        },
    }

    monitoring_config = {
        'selected_model': best_model_name,
        'selected_model_path': str(best_model_path.resolve()),
        'thresholds': {
            'input_drift_psi': 0.2,
            'minimum_f1_score': 0.8 if CONFIG.profile == 'full' else 0.6,
            'minimum_recall': 0.85 if CONFIG.profile == 'full' else 0.6,
        },
        'retraining_policy': {
            'trigger_on_low_recall': True,
            'trigger_on_input_drift': True,
            'evaluation_window_predictions': 250,
        },
    }

    with open(REPORTS_DIR / 'api_contract.json', 'w', encoding='utf-8') as file:
        json.dump(api_contract, file, indent=2)

    with open(CONFIGS_DIR / 'monitoring.json', 'w', encoding='utf-8') as file:
        json.dump(monitoring_config, file, indent=2)

    save_experiment_log([base_metrics, transfer_metrics])

    final_summary = {
        'run_profile': CONFIG.profile,
        'device': str(DEVICE),
        'valid_images': int(len(clean_manifest)),
        'invalid_images': int(len(invalid_images)),
        'best_model': best_model_name,
        'best_model_path': str(best_model_path.resolve()),
        'explainability_figure': str(explainability_path.resolve()),
        'base_cnn_f1': float(base_metrics['f1_score']),
        'transfer_f1': float(transfer_metrics['f1_score']),
    }
    with open(REPORTS_DIR / 'run_summary.json', 'w', encoding='utf-8') as file:
        json.dump(final_summary, file, indent=2)

    print(json.dumps(final_summary, indent=2))
    return final_summary




## Ejecución

Al dejar la llamada al final, el mismo archivo funciona como script y como
notebook generado. Esto mantiene un único flujo operativo.



In [21]:
RUN_RESULTS = run_pipeline()


,split_name,label_name,count
0,test,NORMAL,158
1,test,PNEUMONIA,428
2,train,NORMAL,1266
3,train,PNEUMONIA,3418
4,val,NORMAL,159
5,val,PNEUMONIA,427


Imágenes válidas: 5856
Imágenes inválidas: 0


,sample_id,label_name,split_name,isolation_score
1294,img_05328,NORMAL,train,-0.115248
559,img_00291,NORMAL,train,-0.110856
1064,img_02223,PNEUMONIA,train,-0.080125
1237,img_02099,PNEUMONIA,train,-0.049351
53,img_01915,PNEUMONIA,val,-0.044303
356,img_03029,PNEUMONIA,train,-0.042755
1417,img_01530,PNEUMONIA,train,-0.040683
95,img_01911,PNEUMONIA,train,-0.032495
439,img_04071,PNEUMONIA,train,-0.028112
1055,img_05327,NORMAL,train,-0.025751


Autoencoder epoch 1/2: loss=0.0686
Autoencoder epoch 2/2: loss=0.0624


,sample_id,label_name,reconstruction_error
252,img_01748,PNEUMONIA,0.091252
155,img_00152,NORMAL,0.089563
416,img_02123,PNEUMONIA,0.083699
221,img_01763,PNEUMONIA,0.083243
22,img_02148,PNEUMONIA,0.082881
74,img_04509,PNEUMONIA,0.081841
186,img_04855,PNEUMONIA,0.081698
78,img_05008,PNEUMONIA,0.079667
478,img_00420,NORMAL,0.079226
19,img_01766,PNEUMONIA,0.078846


base_cnn epoch 1/2 | train_loss=0.4083 val_loss=0.1814 val_acc=0.8164 val_f1=0.8589
base_cnn epoch 2/2 | train_loss=0.1966 val_loss=0.1628 val_acc=0.8594 val_f1=0.8947

base_cnn classification report
              precision    recall  f1-score   support

           0     0.9077    0.8551    0.8806        69
           1     0.9476    0.9679    0.9577       187

    accuracy                         0.9375       256
   macro avg     0.9277    0.9115    0.9191       256
weighted avg     0.9369    0.9375    0.9369       256

densenet121_head epoch 1/1 | train_loss=0.3307 val_loss=0.2765 val_acc=0.8789 val_f1=0.9182
densenet121_finetune epoch 1/1 | train_loss=0.1673 val_loss=0.1269 val_acc=0.8828 val_f1=0.9138

densenet121_transfer classification report
              precision    recall  f1-score   support

           0     0.9388    0.6667    0.7797        69
           1     0.8889    0.9840    0.9340       187

    accuracy                         0.8984       256
   macro avg     0.9138

,model_name,threshold,accuracy,precision,recall,f1_score,roc_auc
0,base_cnn,0.20,0.937500,0.947644,0.967914,0.957672,0.963032
1,densenet121_transfer,0.21,0.898438,0.888889,0.983957,0.934010,0.972875


{
  "run_profile": "verify",
  "device": "cpu",
  "valid_images": 5856,
  "invalid_images": 0,
  "best_model": "base_cnn",
  "best_model_path": "C:\\Users\\TeaDyDy\\Documents\\New project\\artifacts\\models\\base_cnn.pt",
  "explainability_figure": "C:\\Users\\TeaDyDy\\Documents\\New project\\artifacts\\figures\\base_cnn_gradcam_comparison.png",
  "base_cnn_f1": 0.9576719576719577,
  "transfer_f1": 0.934010152284264
}
